Импортирую все, что использовал

In [ ]:
import requests
import os
import time
import numpy as np
import json
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

Указываю ссылку на "базу знаний", на всякий случай указал хедер

In [ ]:
base_link = "https://academy.lamoda.ru"
HEADERS = {"User-Agent": "Mozilla/5.0"}

Начинаю создавать сам парсер.
Почти все ссылки на разделы есть на главной странице, можно сделать 1 запрос, чтобы вытащить структуру.
В процессе выяснилось, что некоторые статьи не попадают на главную страницу и становятся доступными при открытии, поэтому нужно будет заходить в разделы и добирать недостающие статьи

По итогу среди собранных ссылок есть и разделы и статьи. Раздел отличается от статьи тем, что его URL является частью другого URL.
Например /articles/api/ раздел, потому что существует /articles/api/1-vvedenie/. Листовые статьи это те у которых нет дочерних ссылок, то есть реальный контент, который нам нужен.

In [ ]:
try:
    main_page = BeautifulSoup(requests.get(base_link, headers=HEADERS).text, "html.parser")
    all_pages_with_links = set()

    for a in main_page.find_all("a", href=True):
        if a['href'].startswith("/articles/"):
            all_pages_with_links.add(a['href'])
except Exception as e:
    print(f"ошибка загрузки страницы: {e}")


for link in list(all_pages_with_links):
    print(f"обрабатываю раздел: {link}")
    try:
        page = BeautifulSoup(requests.get(base_link + link, headers=HEADERS).text, "html.parser")
        for a in page.find_all("a", href=True):
            if a['href'].startswith("/articles/"):
                all_pages_with_links.add(a['href'])
    except Exception as e:
        print(f"ошибка загрузки раздела {link}: {e}")
    time.sleep(0.3)

leaf_pages = []
for link in all_pages_with_links:
    is_section = any(other != link and other.startswith(link) for other in all_pages_with_links)
    if not is_section:
        leaf_pages.append(link)

print(f"найдено статей: {len(leaf_pages)}")

текст статьи всегда находится в div.article-detail__text. Это выяснил экспериментально, проверив HTML структуру нескольких страниц.
Сохраняем в .md формате с указанием источника для последующего использования в ретривере.

In [ ]:
os.makedirs("articles", exist_ok=True)

for link in sorted(leaf_pages):
    try:
        page = BeautifulSoup(requests.get(base_link + link, headers=HEADERS).text, "html.parser")
        article_text = page.find("div", class_="article-detail__text")

        if article_text: # проверяю, что div нашёлся
            filename = link.strip("/").replace("/", "_") + ".md"
            with open(f"articles/{filename}", "w", encoding="utf-8") as f:
                f.write(f"Источник: {base_link}{link}\n\n")
                f.write(article_text.get_text(separator="\n", strip=True))
            print(f"страница {link} обработана")
        else:
            print(f"пусто: {link}")

    except Exception as e:
        print(f"ошибка обработки {link}: {e}")

    time.sleep(0.5)

total = os.listdir("articles")
print(f"всего статей: {len(total)}")

Загружаю все сохранённые статьи в список словарей. Каждый элемент содержит текст и имя файла (источник)

Для поиска использую самый обычный bm25. Дальше разбиваю текст каждой статьи на токены и привожу к нижнему регистру, чтобы один текст с верхним и нижнем регистриом считались одним словом, а затем разбиваю по пробелам.

Дальше пишу функцию поиска: токенизирую запрос, считаю score для каждой статьи и возвращаю топ-k. Score показывает насколько часто слова запроса встречаются в статье с учётом их редкости во всей коллекции.

In [ ]:
all_docs = []
for file in os.listdir("articles"):
    with open(f"articles/{file}", "r", encoding="utf-8") as f:
        text = f.read()
        all_docs.append({"text": text, "source": file})

print(f"загружено статей: {len(all_docs)}")

tokenizer_docs = []
for doc in all_docs:
    tokens = doc["text"].lower().split()
    tokenizer_docs.append(tokens)

bm25 = BM25Okapi(tokenizer_docs)

def search_bm25(query: str, top_k=3) -> list:
    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)
    top_index = np.argsort(scores)[::-1][:top_k]

    results = []
    for i in top_index:
        results.append({
            "source": all_docs[i]["source"],
            "score": scores[i],
            "text": all_docs[i]["text"][:100]
        })
    return results

Теперь поиск по емббедингам. Для построения их выбрал сберовский берт, который адаптирован под наш язык + у меня он уже был скачен.
Кодируем все статьи в векторы (287 × 1024, где 287 это кол-во статей, которые спарсились) и храним в памяти. При поиске кодируем запрос в вектор и считаем косинусное сходство между ним и каждой статьёй. Чем меньше угол между векторами — тем более похожи тексты по смыслу.

In [ ]:
model = SentenceTransformer('ai-forever/sbert_large_nlu_ru')

texts = [doc["text"] for doc in all_docs]
text_embeddings = model.encode(texts, show_progress_bar=True)

def search_dense(query: str, top_k=3) -> list:
    query_embeddings = model.encode([query])
    scores = cosine_similarity(query_embeddings, text_embeddings)[0]
    top_index = np.argsort(scores)[::-1][:top_k]

    result = []
    for i in top_index:
        result.append({
            "source": all_docs[i]["source"],
            "score": scores[i],
            "text": all_docs[i]["text"][:100]
        })

    return result


Теперь гибрид двух методов.
Запускаем оба метода на всех 287 статьях. Получаем два набора scores, но их нельзя складывать напрямую так как они в разных диапазонах (BM25 возвращает числа от 0 до ~10, а косинусы от 0 до 1.). Нормализуем каждый в [0, 1] делением на максимум, затем складываем с равными весами. Сортируем по итоговому score и возвращаем топ-k статей.

In [ ]:
def search_hybrid(query: str, top_k=3, bm25_weight=0.5, dense_weight=0.5) -> list:
    # берём все статьи, чтобы нормализовать scores
    bm25_results = search_bm25(query, top_k=len(all_docs))
    dense_results = search_dense(query, top_k=len(all_docs))

    # превращаем списки в словари {имя_файла: score} для удобного доступа
    bm25_scores = {r["source"]: r["score"] for r in bm25_results}
    dense_scores = {r["source"]: r["score"] for r in dense_results}

    # находим максимум каждого метода для нормализации
    bm25_max = max(bm25_scores.values())
    dense_max = max(dense_scores.values())

    combined = {}
    for source in bm25_scores:
        b_score = bm25_scores[source] / bm25_max if bm25_max > 0 else 0
        d_score = dense_scores[source] / dense_max if dense_max > 0 else 0

        # итоговый score = взвешенная сумма, веса равны по 0.5
        combined[source] = bm25_weight * b_score + dense_weight * d_score

    # сортируем по итоговому score и берём топ-k
    top_sources = sorted(combined, key=combined.get, reverse=True)[:top_k]

    # ищем индекс статьи в all_docs чтобы достать текст
    results = []
    for source in top_sources:
        idx = next(i for i, d in enumerate(all_docs) if d["source"] == source)
        results.append({
            "source": source,
            "score": combined[source],
            "text": all_docs[idx]["text"][:200]
        })
    return results

Теперь агент:
Использую Llama 3.1, запущенную локально через Ollama.
Теперь прописываю инструмент и переходим к запуску агента. Я дал достаточно примитивный запрос, который модель будет читать при каждом вызове, то есть инструкции по сути.
После этого цикл, который будет описывать модель поведения, если можно так сказать: моделька думает -> вызывает инструмент -> думает -> отвечает.

Если модель решает вызывать инструмент, то логируем это действие в истории. Затем достаем запрос, который модель сформировала, делаем вызов гибридного поиска, чтобы отдать релевантный ответ и форматируем ответ для модели. Модель дала ответ и выходим из цикла

In [ ]:
client = OpenAI(
    api_key="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",
    base_url="https://api.groq.com/openai/v1"
)


tools = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": "Ищет информацию в базе знаний академии селлеров Lamoda",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Поисковый запрос"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

def run_agent(user_question):
    messages = [
        {"role": "system", "content":
        "Ты помощник по академии селлеров Lamoda. Твоя задача дать максимально релевантный"
        "ответ для пользователя. Для ответа на вопросы используй инструмент"
        "search_knowledge_base. В конце ответа указывай источники."},
        {"role": "user", "content": user_question}
    ]


    while True:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            tools=tools
        )

        message = response.choices[0].message


        if message.tool_calls:
            messages.append(message)

            for tool_call in message.tool_calls:
                args = json.loads(tool_call.function.arguments)
                query = args["query"]
                print(f"Поиск: {query}")

                results = search_hybrid(query, top_k=3)

                tool_result = "\n\n".join([
                    f"Источник: {r['source']}\n{r['text']}"
                    for r in results
                ])

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result
                })

        else:
            print(f"\nОтвет: {message.content}")
            break

run_agent("Как зарегистрироваться в Lamoda Seller?")
run_agent("Как добавить товар в акцию?")
run_agent("Что такое FBO и как начать работу?")

Планы по улучшению:

1) Самый простой, думаю стоит добавить чанкование, так как сейчас моделька для эмббедингов обрабатывает 512 токенов, а все что длиннее обрезается, таким образом мы теряем информацию. Можно нарезать 150-300 слов. Или меньше, чтобы улушить качество ответов. Так же стоит добавить перекрытие, чтобы не терять информацию, которая растягивается, например, на несколько предложений.

2) Сохранить емббединги, чтобы их все время не пересчитывать, таким образом оптимизируем время.

3) Думаю можно попробовать считать скоры другими путями, а не линейными комбинациями. Слышал об Reciprocal Rank Fusion, метод объединяет несколько ранжированных списков в один отсортированный с их скорами, затем берём топ-k и отдаём агенту.